# RDFSolve Inference Analysis

This notebook analyzes the class derivation and instance matching results:

1. Instance-Level Mappings Analysis
2. Class Derivation Results
3. SeMRA/SSSOM Integration
4. Inferred Connectivity Graph

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns
from IPython.display import display

from rdfsolve import (
    ClassIndex,
    ClassPair,
    MappingEdge,
    derive_class_mappings,
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Configuration
OUTPUT_DIR = Path("/home/javier.millanacosta/rdfsolve/output")
MAPPINGS_DIR = OUTPUT_DIR / "mappings"
FIGURES_DIR = Path("./figures")
FIGURES_DIR.mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Mappings directory: {MAPPINGS_DIR}")

## 1. Load Mapping Data

In [ ]:
def load_mapping_files(mappings_dir: Path) -> dict:
    """Load all mapping files from subdirectories."""
    results = {}
    
    for subdir in ["sssom", "semra", "instance_matching", "class_derived", "inferenced"]:
        dir_path = mappings_dir / subdir
        results[subdir] = []
        
        if not dir_path.exists():
            continue
        
        for f in dir_path.glob("*.jsonld"):
            try:
                data = json.loads(f.read_text())
                edges = data.get("@graph", [])
                
                results[subdir].append({
                    "file": f.name,
                    "path": f,
                    "data": data,
                    "edges": edges,
                    "edge_count": len(edges),
                    "about": data.get("@about", {}),
                })
            except Exception as e:
                print(f"Error loading {f}: {e}")
    
    return results


mappings = load_mapping_files(MAPPINGS_DIR)

print("Mapping files loaded:")
for source, files in mappings.items():
    total_edges = sum(f["edge_count"] for f in files)
    print(f"  {source}: {len(files)} files, {total_edges:,} edges")

## 2. Analyze SSSOM Mappings

In [ ]:
# Analyze SSSOM mappings
sssom_stats = []

for f in mappings.get("sssom", []):
    edges = f["edges"]
    
    # Count predicates
    predicates = Counter()
    source_prefixes = Counter()
    target_prefixes = Counter()
    
    for edge in edges:
        predicates[edge.get("predicate", "unknown")] += 1
        
        source = edge.get("source_class", "")
        target = edge.get("target_class", "")
        
        if ":" in source:
            source_prefixes[source.split(":")[0]] += 1
        if ":" in target:
            target_prefixes[target.split(":")[0]] += 1
    
    sssom_stats.append({
        "file": f["file"],
        "edges": len(edges),
        "predicates": dict(predicates),
        "top_source_prefixes": dict(source_prefixes.most_common(5)),
        "top_target_prefixes": dict(target_prefixes.most_common(5)),
    })

if sssom_stats:
    print("SSSOM Mapping Summary:")
    for stat in sssom_stats[:5]:
        print(f"\n{stat['file']}:")
        print(f"  Edges: {stat['edges']}")
        print(f"  Predicates: {stat['predicates']}")
else:
    print("No SSSOM mappings found")

## 3. Analyze SeMRA Mappings

In [ ]:
# Analyze SeMRA mappings
semra_stats = []

for f in mappings.get("semra", []):
    edges = f["edges"]
    
    predicates = Counter()
    confidence_scores = []
    
    for edge in edges:
        predicates[edge.get("predicate", "unknown")] += 1
        
        conf = edge.get("confidence")
        if conf is not None:
            confidence_scores.append(float(conf))
    
    semra_stats.append({
        "file": f["file"],
        "edges": len(edges),
        "predicates": dict(predicates),
        "avg_confidence": sum(confidence_scores) / len(confidence_scores) if confidence_scores else None,
    })

if semra_stats:
    print("SeMRA Mapping Summary:")
    df_semra = pd.DataFrame(semra_stats).sort_values("edges", ascending=False)
    display(df_semra.head(10))
else:
    print("No SeMRA mappings found")

## 4. Analyze Instance Matching Results

In [ ]:
# Analyze instance matching
instance_stats = []

for f in mappings.get("instance_matching", []):
    edges = f["edges"]
    about = f["about"]
    
    datasets_involved = set()
    for edge in edges:
        datasets_involved.add(edge.get("source_dataset", "unknown"))
        datasets_involved.add(edge.get("target_dataset", "unknown"))
    
    instance_stats.append({
        "file": f["file"],
        "edges": len(edges),
        "datasets": len(datasets_involved),
        "resource_prefix": about.get("resource_prefix", "unknown"),
    })

if instance_stats:
    print("Instance Matching Summary:")
    df_instance = pd.DataFrame(instance_stats).sort_values("edges", ascending=False)
    display(df_instance)
    
    # Visualize
    if len(df_instance) > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(df_instance["resource_prefix"][:15], df_instance["edges"][:15])
        ax.set_xlabel("Number of Edges")
        ax.set_title("Instance Matches by Resource Prefix")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "instance_matches.png", dpi=300, bbox_inches='tight')
        plt.show()
else:
    print("No instance matching results found")

## 5. Analyze Class Derivation Results

In [ ]:
# Analyze class derived mappings
class_derived_stats = []

for f in mappings.get("class_derived", []):
    edges = f["edges"]
    
    confidence_scores = []
    instance_counts = []
    
    for edge in edges:
        conf = edge.get("confidence")
        inst = edge.get("instance_count")
        
        if conf is not None:
            confidence_scores.append(float(conf))
        if inst is not None:
            instance_counts.append(int(inst))
    
    class_derived_stats.append({
        "file": f["file"],
        "edges": len(edges),
        "avg_confidence": sum(confidence_scores) / len(confidence_scores) if confidence_scores else None,
        "avg_instance_count": sum(instance_counts) / len(instance_counts) if instance_counts else None,
        "max_instance_count": max(instance_counts) if instance_counts else None,
    })

if class_derived_stats:
    print("Class Derivation Summary:")
    df_class = pd.DataFrame(class_derived_stats).sort_values("edges", ascending=False)
    display(df_class)
    
    # Confidence distribution
    all_confidences = []
    for f in mappings.get("class_derived", []):
        for edge in f["edges"]:
            conf = edge.get("confidence")
            if conf is not None:
                all_confidences.append(float(conf))
    
    if all_confidences:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.hist(all_confidences, bins=50, edgecolor='black', alpha=0.7)
        ax.set_xlabel("Confidence Score")
        ax.set_ylabel("Count")
        ax.set_title("Class Derivation Confidence Distribution")
        ax.axvline(x=0.5, color='red', linestyle='--', label='Threshold 0.5')
        ax.legend()
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "class_derivation_confidence.png", dpi=300, bbox_inches='tight')
        plt.show()
else:
    print("No class derived mappings found")

## 6. Build Inferred Connectivity Graph

In [ ]:
# Build a graph of dataset connectivity based on all mappings
G = nx.Graph()

# Add edges from all mapping types
edge_counts = defaultdict(lambda: defaultdict(int))

for source_type, files in mappings.items():
    for f in files:
        for edge in f["edges"]:
            src_ds = edge.get("source_dataset", "unknown")
            tgt_ds = edge.get("target_dataset", "unknown")
            
            if src_ds and tgt_ds and src_ds != tgt_ds:
                # Normalize order
                pair = tuple(sorted([src_ds, tgt_ds]))
                edge_counts[pair][source_type] += 1

# Add to graph
for (src, tgt), type_counts in edge_counts.items():
    total = sum(type_counts.values())
    G.add_edge(src, tgt, weight=total, **type_counts)

print(f"Connectivity Graph:")
print(f"  Nodes (datasets): {G.number_of_nodes()}")
print(f"  Edges (connections): {G.number_of_edges()}")

if G.number_of_nodes() > 0:
    # Connected components
    components = list(nx.connected_components(G))
    print(f"  Connected components: {len(components)}")
    print(f"  Largest component: {len(max(components, key=len))} nodes")
    
    # Top connected datasets
    degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)
    print(f"\nTop 10 connected datasets:")
    for name, degree in degrees[:10]:
        print(f"  {name}: {degree} connections")

In [ ]:
# Visualize connectivity graph
if G.number_of_nodes() > 0 and G.number_of_edges() > 0:
    fig, ax = plt.subplots(figsize=(16, 12))
    
    # Use largest component for cleaner visualization
    if len(components) > 1:
        largest = max(components, key=len)
        subgraph = G.subgraph(largest)
    else:
        subgraph = G
    
    # Layout
    pos = nx.spring_layout(subgraph, k=1.5, iterations=50, seed=42)
    
    # Node sizes by degree
    sizes = [100 + subgraph.degree(n) * 50 for n in subgraph.nodes()]
    
    # Edge widths by weight
    weights = [min(5, subgraph[u][v].get('weight', 1) / 100) for u, v in subgraph.edges()]
    
    # Draw
    nx.draw_networkx_nodes(subgraph, pos, node_size=sizes, node_color='lightblue', alpha=0.8, ax=ax)
    nx.draw_networkx_edges(subgraph, pos, width=weights, alpha=0.3, ax=ax)
    nx.draw_networkx_labels(subgraph, pos, font_size=7, ax=ax)
    
    ax.set_title("Inferred Dataset Connectivity (Based on All Mappings)")
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "inferred_connectivity.png", dpi=300, bbox_inches='tight')
    plt.savefig(FIGURES_DIR / "inferred_connectivity.pdf", bbox_inches='tight')
    plt.show()
else:
    print("No graph data to visualize")

## 7. Export Results

In [ ]:
# Export summary statistics
inference_summary = {
    "mapping_sources": {
        source: {
            "files": len(files),
            "total_edges": sum(f["edge_count"] for f in files),
        }
        for source, files in mappings.items()
    },
    "connectivity": {
        "datasets": G.number_of_nodes(),
        "connections": G.number_of_edges(),
        "connected_components": len(components) if 'components' in dir() else 0,
    },
    "sssom_stats": sssom_stats[:5] if sssom_stats else [],
    "instance_stats": instance_stats[:5] if instance_stats else [],
    "class_derived_stats": class_derived_stats[:5] if class_derived_stats else [],
}

summary_path = OUTPUT_DIR / "inference_summary.json"
summary_path.write_text(json.dumps(inference_summary, indent=2, default=str), encoding="utf-8")
print(f"Saved inference summary to {summary_path}")

# List figures
print("\nGenerated figures:")
for f in sorted(FIGURES_DIR.glob("*")):
    print(f"  - {f.name}")